<a href="https://colab.research.google.com/github/shubhanagrawal/Machine-Learning-and-Data-Analysis/blob/main/end_to_end_medical_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install langchain langchain-pinecone langchain-huggingface pinecone-client[grpc] transformers torch python-dotenv pypdf langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 763.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.6/304.6 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [3]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [4]:
#Extract Data From the PDF File
from langchain.document_loaders import PyPDFLoader

def load_pdf_file(data):
    loader= PyPDFLoader(data)
    documents=loader.load()
    return documents

In [5]:
extracted_data=load_pdf_file(data=r'/content/Medical_book.pdf')

In [ ]:
extracted_data

In [ ]:
#Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [ ]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

In [ ]:
text_chunks

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:
#Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings


In [ ]:
embeddings = download_hugging_face_embeddings()

In [ ]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

In [ ]:
query_result

In [ ]:
!pip install python-dotenv


In [ ]:
from google.colab import userdata
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')

In [ ]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

In [ ]:
!pip install Pinecone

In [ ]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medicalbot"


pc.delete_index(index_name)
pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)


In [ ]:
# Embed each chunk and upsert the embeddings into your Pinecone index.
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings,
)

In [ ]:
# Load Existing index

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [ ]:
docsearch

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
retrieved_docs = retriever.invoke("What is Acne?")

In [ ]:
retrieved_docs

In [ ]:
!pip install transformers

In [ ]:
from langchain.chains.question_answering import load_qa_chain
from langchain import HuggingFacePipeline

In [ ]:
from transformers import pipeline

# Initialize a text2text-generation pipeline with a Flan-T5 model
# You can choose a different size model depending on your needs and available resources.
# For example: google/flan-t5-small, google/flan-t5-base, google/flan-t5-large, etc.
qa_pipeline = pipeline("text2text-generation", model="google/flan-t5-base")

llm = HuggingFacePipeline(pipeline=qa_pipeline)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "What is stats?"})
print(response["answer"])

In [ ]:
from langchain.chains import RetrievalQA

# Create a question answering chain using RetrievalQA
qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)

In [ ]:
# Ask a question and get an answer from the chain
query = "What is stats?"
answer = qa_chain.invoke({"query": query})

In [ ]:
print(answer['result'])

In [ ]:
from langchain.chains import RetrievalQA

Streamlit App




In [ ]:
# Review of app.py imports:
# streamlit
# langchain.document_loaders.PyPDFLoader
# langchain.text_splitter.RecursiveCharacterTextSplitter
# langchain.embeddings.HuggingFaceEmbeddings
# pinecone.grpc.PineconeGRPC
# pinecone.ServerlessSpec
# langchain_pinecone.PineconeVectorStore
# langchain.chains.RetrievalQA
# langchain.HuggingFacePipeline
# transformers.pipeline
# os

# Create requirements.txt
requirements_content = """streamlit
langchain
langchain-pinecone
langchain-huggingface
pinecone-client[grpc]
transformers
torch
python-dotenv
pypdf
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content)

print("requirements.txt created successfully.")

In [18]:
%%writefile app.py
import streamlit as st
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain.chains import RetrievalQA # Keep for now, might be removed later
from langchain import HuggingFacePipeline
from transformers import pipeline
import os
from pathlib import Path
# Removed: from google.colab import userdata # Import userdata

# Added for the new prompt
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


# Code for loading and splitting PDF
def load_pdf_file(file_path):
    loader = PyPDFLoader(str(file_path))
    documents = loader.load()
    return documents

def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

def setup_pinecone(index_name="medicalbot", dimension=384, cloud="aws", region="us-east-1", api_key=None):
    if api_key is None:
        st.error("Pinecone API key not found. Please set the PINECONE_API_KEY environment variable.")
        st.stop()

    pc = Pinecone(api_key=api_key)

    try:
        if index_name not in pc.list_indexes():
            pc.create_index(
                name=index_name,
                dimension=dimension,
                metric="cosine",
                spec=ServerlessSpec(cloud=cloud, region=region)
            )
            st.info(f"Index '{index_name}' created successfully.")
        else:
            st.info(f"Index '{index_name}' already exists.")
    except Exception as e:
        if "ALREADY_EXISTS" in str(e):
            st.warning(f"Pinecone index '{index_name}' already exists. Proceeding with existing index.")
        else:
            st.error(f"Failed to create or access index '{index_name}': {e}")
            st.stop()

    return pc


def setup_vector_store(text_chunks, embeddings, index_name, pc):
    index = pc.Index(index_name)
    index_stats = index.describe_index_stats()
    if index_stats.total_vector_count == 0:
        docsearch = PineconeVectorStore.from_documents(
            documents=text_chunks,
            index_name=index_name,
            embedding=embeddings,
        )
    else:
        docsearch = PineconeVectorStore.from_existing_index(
            index_name=index_name,
            embedding=embeddings
        )
    retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})
    return retriever

def initialize_llm():
    qa_pipeline = pipeline("text2text-generation", model="google/flan-t5-base")
    llm = HuggingFacePipeline(pipeline=qa_pipeline)
    return llm

def setup_qa_chain(llm, retriever):
    # Define the prompt template within the setup function or globally if preferred
    system_prompt = (
        "You are an assistant for question-answering tasks. "
        "Use the following pieces of retrieved context to answer "
        "the question. If you don't know the answer, say that you "
        "don't know. Use three sentences maximum and keep the "
        "answer concise."
        "\n\n"
        "{context}"
    )
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            ("human", "{input}"),
        ]
    )

    # Create the chains using the new prompt
    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, question_answer_chain)
    return rag_chain # Return the new rag_chain


# Streamlit App
if __name__ == "__main__":
    st.title("🩺 Medical Question Answering Bot")

    # API Key: Read from environment variable
    PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")

    if not PINECONE_API_KEY:
        st.error("Pinecone API key not found. Please set the PINECONE_API_KEY environment variable.")
        st.stop()


    # PDF File (ensure this exists)
    pdf_path = Path("Medical_book.pdf")
    if not pdf_path.exists():
        st.error("The PDF file was not found. Please make sure 'Medical_book.pdf' exists.")
        st.stop()

    try:
        extracted_data = load_pdf_file(pdf_path)
        text_chunks = text_split(extracted_data)
        st.success(f"PDF loaded and split into {len(text_chunks)} chunks.")
    except Exception as e:
        st.error(f"Error loading or splitting PDF: {e}")
        st.stop()

    try:
        embeddings = download_hugging_face_embeddings()
        st.success("Embeddings downloaded.")
    except Exception as e:
        st.error(f"Error downloading embeddings: {e}")
        st.stop()

    index_name = "medicalbot"
    try:
        pc = setup_pinecone(api_key=PINECONE_API_KEY, index_name=index_name)
        retriever = setup_vector_store(text_chunks, embeddings, index_name, pc)
        st.success("Pinecone index and vector store set up.")
    except Exception as e:
        st.error(f"Error setting up Pinecone or vector store: {e}")
        st.stop()

    try:
        llm = initialize_llm()
        st.success("Language model initialized.")
    except Exception as e:
        st.error(f"Error initializing language model: {e}")
        st.stop()

    try:
        # Updated to use the new chain setup
        qa_chain = setup_qa_chain(llm, retriever)
        st.success("QA chain set up successfully.")
    except Exception as e:
        st.error(f"Error setting up QA chain: {e}")
        st.stop()

    st.markdown("### Ask your medical question below 👇")
    query = st.text_input("Enter your question:")
    get_answer_button = st.button("Get Answer")

    if get_answer_button:
        if not query.strip():
            st.warning("Please enter a question.")
        else:
            with st.spinner("Searching for answer..."):
                try:
                    # Updated to use invoke with the new chain
                    response = qa_chain.invoke({"input": query})
                    st.markdown("**Answer:**")
                    # The response structure might be different with the new chain
                    # Assuming the answer is still in 'answer' or 'result' key
                    st.write(response.get("answer", response.get("result", "No result returned.")))
                except Exception as e:
                    st.error(f"Error getting answer: {e}")

Overwriting app.py


In [9]:
import os
from google.colab import userdata

# Get the API key from Colab secrets
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')

# Set the API key as an environment variable and run the Streamlit app
os.environ['PINECONE_API_KEY'] = PINECONE_API_KEY

# !streamlit run app.py --server.port 8501 & npx localtunnel --port 8501

In [ ]:
!streamlit run app.py --server.port 8501 & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.90.229.140:8501

⠧your url is: https://great-ducks-listen.loca.lt
/content/app.py:2: LangChainDeprecationWarning: Importing PyPDFLoader from langchain.document_loaders is deprecated. Please replace deprecated imports:

>> from langchain.document_loaders import PyPDFLoader

with new imports of:

>> from langchain_community.document_loaders import PyPDFLoader
You can use the langchain cli to **automatically** upgrade many imports. Please see documentation here <https://python.langchain.com/docs/versions/v0_2/>
  from langchain.document_loaders import PyPDFLoader
/content/app.py:4: LangChainDeprecationWarning: Importing HuggingFaceEmbeddings from langchain.embeddings is deprecated. Please replace deprecated imports:

>> from langchain.embeddings import HuggingFaceEmbeddings

with new imports of:

>> from langchain_community.em

In [8]:
!pip install streamlit langchain langchain-pinecone langchain-huggingface pinecone-client[grpc] transformers torch python-dotenv pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.6/304.6 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [14]:
!pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00


In [ ]:
!streamlit run app.py --server.port 8501 & npx localtunnel --port 8501